# Близкие фразы: проверяем косинус

Дополнение к § 3.3. Используем маленькие учебные векторы, чтобы понимать ранжирование до запуска большой языковой модели.

## Сходство и направление
Проверьте сходство запроса с тремя кандидатами, включая противоположный.

In [ ]:
import numpy as np
query = np.array([1.0,0.0])
candidates = np.array([[0.8,0.6],[0,1],[-1,0]], dtype=float)
def cosine(a,b):
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b)))
scores = [cosine(query,row) for row in candidates]
print(np.round(scores,3))
assert np.allclose(scores, [0.8,0,-1])

## Изменяем запрос
Поменяйте query на (0,1). Какие кандидаты переместились? Сходство не является вероятностью.

In [ ]:
new_query = np.array([0.0,1.0])
new_scores = [cosine(new_query,row) for row in candidates]
print(sorted(enumerate(new_scores), key=lambda x: (-x[1],x[0])))

## Проверка на своих фразах
После запуска модели из статьи запишите русскую и белорусскую пару и фразу с отрицанием; сравните смысл и порядок соседей.

In [ ]:
phrases = [("Когда забрать фигурку?", "Калі забраць фігурку?"),
           ("Когда забрать фигурку?", "Не нужно забирать фигурку")]
for left,right in phrases:
    print(left, "→", right)

## Опыт с русским и белорусским текстом

Запишите гипотезу для двух пар фраз: перефразирование и отрицание. Установите зависимости статьи и включите run_model. Классификатор применяйте к русским отзывам; многоязычную модель предложений проверяйте отдельно на обоих языках.


In [ ]:
run_model = False
if run_model:
    import pandas as pd
    from transformers import pipeline
    from sentence_transformers import SentenceTransformer

    reviews = [
        "Фигурка получилась красивой",
        "Фигурка не получилась красивой",
        "Фигурку напечатали вчера",
        "Отлично, снова сломался принтер",
    ]
    classifier = pipeline(
        "text-classification",
        model="blanchefort/rubert-base-cased-sentiment",
    )
    outputs = classifier(reviews)
    report = pd.DataFrame({
        "text": reviews,
        "label": [x["label"] for x in outputs],
        "score": [x["score"] for x in outputs],
    })
    print(report.to_string(index=False))

    phrases = [
        "Когда забрать фигурку?",
        "Калі забраць фігурку?",
        "Не нужно забирать фигурку",
        "Где получить готовую модель?",
    ]
    encoder = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    vectors = encoder.encode(phrases, normalize_embeddings=True)
    similarity = vectors @ vectors.T
    print(pd.DataFrame(similarity, index=phrases, columns=phrases).round(2))


## Разбор результата

Для каждого русского отзыва запишите человеческую метку. Если смысл ироничной фразы спорный, не включайте её в простую долю верных без правила разметки. В матрице сходства сравните русско-белорусскую пару и пару с отрицанием. Поясните, когда высокая близость полезна, а когда она опасна.


In [ ]:
if run_model:
    print("Классы модели:", classifier.model.config.id2label)
    print("Косинус русского и белорусского вопроса:",
          round(float(similarity[0, 1]), 3))
    print("Косинус фразы с отрицанием:",
          round(float(similarity[0, 2]), 3))


## Самопроверка




### 1. На какой вопрос отвечает классификатор тональности?

A. Истинно ли сообщение.

B. Какую метку отношения выбрать.

C. Где находится предмет.

D. Как перевести текст.




### 2. Почему фраза «Отлично, снова сломался принтер» трудна?

A. В ней нет букв.

B. Она может быть ироничной.

C. Она слишком длинна.

D. В ней нет пробелов.




### 3. Что возвращает модель эмбеддингов для фразы?

A. Рамку.

B. Вектор чисел.

C. Новые веса.

D. Только знак препинания.




### 4. Чему равно косинусное сходство (1,0) и (0,1)?

A. −1.

B. 0.

C. 0,5.

D. 1.




### 5. Можно ли считать косинус 0,8 вероятностью смыслового совпадения?

A. Да.

B. Только для русского.

C. Нет.

D. Только при двух фразах.




### 6. Зачем нормировать эмбеддинги перед произведением?

A. Чтобы получить косинус обычным скалярным произведением.

B. Чтобы перевести текст.

C. Чтобы удалить отрицание.

D. Чтобы изменить веса.




### 7. Почему русскоязычный классификатор нужно отдельно проверять на белорусском?

A. Алфавиты полностью разные.

B. Поддержка русского не доказывает качество на другом языке.

C. Белорусский не содержит отрицаний.

D. Веса всегда случайные.




### 8. Что делать, если два человека расходятся в метке ироничного отзыва?

A. Скрыть расхождение.

B. Записать неоднозначность и согласовать критерий.

C. Объявить модель правой.

D. Удалить все отзывы.




### 9. Что означает наибольший косинус среди четырёх фраз?

A. Фраза гарантированно синоним.

B. Она ближе остальных в выбранном пространстве.

C. Она самая короткая.

D. Она истинна.




### 10. Что важно сохранять при сравнении двух запусков?

A. Язык, модель и входные фразы.

B. Только цвет текста.

C. Только самый высокий балл.

D. Число открытых вкладок.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Посмотрите на список меток модели.



**Решение**


**B.** Он выбирает тональность из заданных классов.



### Вопрос 2

**Подсказка**


Слова одобрения не всегда означают одобрение.



**Решение**


**B.** Ирония требует контекста и может сбить классификатор и человека.



### Вопрос 3

**Подсказка**


По этому результату вычисляют косинус.



**Решение**


**B.** Эмбеддинг представляет фразу набором координат.



### Вопрос 4

**Подсказка**


Скалярное произведение равно нулю.



**Решение**


**B.** Векторы перпендикулярны, поэтому косинус равен 0.



### Вопрос 5

**Подсказка**


Сходство — геометрическая мера.



**Решение**


**C.** Косинус ранжирует векторы, но не является калиброванной вероятностью.



### Вопрос 6

**Подсказка**


После нормировки длины равны единице.



**Решение**


**A.** Знаменатель формулы косинуса становится равным единице.



### Вопрос 7

**Подсказка**


Область обучения и применения может отличаться.



**Решение**


**B.** Работу на новом языке проверяют на примерах с известным смыслом.



### Вопрос 8

**Подсказка**


Для оценки нужно понятное правило разметки.



**Решение**


**B.** Неоднозначные примеры нельзя выдавать за однозначную ошибку модели.



### Вопрос 9

**Подсказка**


Сравнение относительно кандидатов и модели.



**Решение**


**B.** Лидер по сходству не обязательно точно совпадает по смыслу.



### Вопрос 10

**Подсказка**


Изменение входа может объяснить изменение ответа.



**Решение**


**A.** Сравнение должно фиксировать модель и тексты.



